# 01 — Wage Policy Analysis

**Goal:** Show exactly how Texas Medicaid reimbursement rates constrain direct care worker wages, and what it would take to change that.

**Three Drivers of Low Wages:**
1. Texas sets reimbursement assumptions low ($10.60 → $13 base)
2. Market clears elsewhere (Buc-ee's, Amazon, retail pay more)
3. Provider overhead + compliance eats remaining margin

**Data Sources:**
- PFD Personal Attendant Base Wage Calculator (HHSC's own spreadsheet)
- BLS OEWS May 2024 (Texas)
- Market wage comparisons from public reporting
- KERA, Texas Standard, Texas Tribune coverage

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from texas_hhcs.staffing import StaffingModel, compare_wages
from pathlib import Path

PROCESSED = Path('../data/processed')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120

---
## 1. The Smoking Gun: HHSC's Own Wage Assumption

HHSC publishes a "Personal Attendant Base Wage Calculator" that literally has
`$10.60/hr` as the default target wage in cell B7. Every Medicaid service rate
is built on top of this number.

Source: PFD Rate Tables → `ltss-personal-attendant-base-wage-calculator.xlsx`

In [ ]:
# Load the processed wage calculator data
df_all = pd.read_csv(PROCESSED / 'pfd_wage_calculator_all_services.csv')
df_target = pd.read_csv(PROCESSED / 'pfd_wage_calculator_target_programs.csv')

print(f"Total service lines in HHSC wage calculator: {len(df_all)}")
print(f"Programs covered: {df_all['program'].nunique()}")
print(f"\nTarget programs (HCS, ICF, TxHmL, CAS, CLASS): {len(df_target)} service lines")

In [ ]:
# What percentage of each rate goes to attendant wages?
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: Attendant cost as % of rate by program
programs_order = ['CAS', 'CLASS', 'HCS', 'TxHmL', 'ICF']
df_plot = df_target[df_target['program'].isin(programs_order)].copy()

sns.boxplot(data=df_plot, x='program', y='attendant_cost_pct', order=programs_order, ax=axes[0])
axes[0].set_title('What % of the Medicaid Rate Goes to Attendant Wages?', fontweight='bold')
axes[0].set_xlabel('Program')
axes[0].set_ylabel('Attendant Cost as % of Rate')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='50% line')
axes[0].legend()

# Right: Distribution of wage assumptions across all services
# Filter to hourly-type services (where wage_required is roughly an hourly figure)
hourly_services = df_target[
    (df_target['wage_required'] > 0) & 
    (df_target['wage_required'] < 25)  # hourly range
].copy()

axes[1].hist(hourly_services['wage_required'], bins=20, edgecolor='black', alpha=0.7, color='steelblue')
axes[1].axvline(x=10.60, color='red', linestyle='--', linewidth=2, label='HHSC default: $10.60')
axes[1].axvline(x=13.00, color='orange', linestyle='--', linewidth=2, label='New target: $13.00')
axes[1].set_title('Distribution of Wage Assumptions Baked Into Rates\n(hourly services only)', fontweight='bold')
axes[1].set_xlabel('Wage Required ($/hr)')
axes[1].set_ylabel('Number of Service Lines')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/wage_assumptions_by_program.png', bbox_inches='tight')
plt.show()

---
## 2. The Market Reality: Who Pays More Than "Life-or-Death" Care?

BLS OEWS May 2024 data for Texas + public employer wage data.

In [ ]:
# Load comparison data
df_market = pd.read_csv(PROCESSED / 'market_wages.csv')
df_bls = pd.read_csv(PROCESSED / 'bls_oews_texas_2024.csv')

# Build the comparison chart
fig, ax = plt.subplots(figsize=(14, 8))

df_plot = df_market.sort_values('wage', ascending=True).copy()

# Color by category
colors = {
    'medicaid_assumption': '#d32f2f',
    'bls_actual': '#1976d2', 
    'retail': '#388e3c',
    'food_service': '#f57c00',
    'minimum': '#757575',
}
bar_colors = [colors.get(cat, '#999') for cat in df_plot['category']]

bars = ax.barh(df_plot['employer'], df_plot['wage'], color=bar_colors, edgecolor='white', height=0.7)

# Add wage labels on bars
for bar, wage in zip(bars, df_plot['wage']):
    ax.text(bar.get_width() + 0.15, bar.get_y() + bar.get_height()/2, 
            f'${wage:.2f}', va='center', fontweight='bold', fontsize=11)

# Add reference lines
ax.axvline(x=10.60, color='#d32f2f', linestyle=':', alpha=0.6)
ax.axvline(x=13.00, color='#d32f2f', linestyle='--', alpha=0.6)

ax.set_xlabel('Hourly Wage ($)', fontsize=13)
ax.set_title('Texas Direct Care Worker Wages vs. Market Alternatives\n'
             '"Making life-or-death decisions for $10.60 an hour" — Texas Standard', 
             fontweight='bold', fontsize=14)
ax.set_xlim(0, 21)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#d32f2f', label='HHSC Medicaid assumption'),
    Patch(facecolor='#1976d2', label='BLS actual (TX, May 2024)'),
    Patch(facecolor='#388e3c', label='Retail'),
    Patch(facecolor='#f57c00', label='Food service'),
    Patch(facecolor='#757575', label='Minimum wage'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.tight_layout()
plt.savefig('../reports/wage_comparison_chart.png', bbox_inches='tight')
plt.show()

In [ ]:
# BLS wage distribution for Home Health Aides in Texas
hha = df_bls[df_bls['soc_code'] == '31-1120'].set_index('measure')['value']

print("=== BLS Texas Home Health & Personal Care Aides (May 2024) ===")
print(f"  Employment:     {hha['employment']:>10,.0f} workers")
print(f"  10th percentile: ${hha['pct10_hourly']:>7.2f}/hr")
print(f"  25th percentile: ${hha['pct25_hourly']:>7.2f}/hr  ← matches HHSC $10.60 almost exactly")
print(f"  Median:          ${hha['median_hourly']:>7.2f}/hr")
print(f"  Mean:            ${hha['hourly_mean']:>7.2f}/hr")
print(f"  Annual mean:     ${hha['annual_mean']:>10,.0f}")
print()
print(f"  → The HHSC $10.60 target is the 25th percentile of MARKET wages.")
print(f"  → Even the $13 'raise' is ABOVE the market mean ($12.19) — but")
print(f"    the market mean is already suppressed by Medicaid rates.")
print(f"  → 314,610 workers in TX at this wage level. This is a massive workforce.")

---
## 3. HCS Residential: The House-Level Math

What does the rate actually buy for a group home?

In [ ]:
# HCS Supervised Living / Residential Support rates from PFD calculator
hcs_res = df_target[
    (df_target['program'] == 'HCS') & 
    (df_target['service_description'].str.contains('Supervised Living|Residential', na=False))
].drop_duplicates(subset=['current_rate', 'attendant_cost']).copy()

# Map LON from bill description
hcs_res['lon'] = hcs_res['bill_description'].str.extract(r'LON (\d+)').astype(int)
hcs_res['service_type'] = hcs_res['bill_description'].apply(
    lambda x: 'Supervised Living' if 'SUPERVISED' in str(x).upper() else 'Residential Support'
)
hcs_res = hcs_res.sort_values('lon')

print("=== HCS Residential/Supervised Living Daily Rates ===")
print(f"{'LON':<6} {'Type':<25} {'Daily Rate':>12} {'Att Cost':>12} {'Att %':>8} {'Wage/Day':>12}")
print('-' * 80)
for _, r in hcs_res.iterrows():
    print(f"LON {r['lon']:<3} {r['service_type']:<25} ${r['current_rate']:>10.2f} "
          f"${r['attendant_cost']:>10.2f} {r['attendant_cost_pct']:>7.1%} ${r['wage_required']:>10.2f}")

In [ ]:
# Model a 4-bed house at LON 5 (common level)
lon5_daily = 158.00  # daily rate per resident
beds = 4
occupancy = 0.95  # realistic occupancy

annual_revenue = lon5_daily * beds * occupancy * 365

# Staffing model at $13/hr
model_13 = StaffingModel(residents=4, staff_per_shift=1, hourly_wage=13.00)
costs_13 = model_13.annual_labor_cost(overtime_hours_pct=0.15)

# Same model at $15/hr (what it should be)
model_15 = StaffingModel(residents=4, staff_per_shift=1, hourly_wage=15.00)
costs_15 = model_15.annual_labor_cost(overtime_hours_pct=0.15)

# Same model at $18/hr (Buc-ee's competitive)
model_18 = StaffingModel(residents=4, staff_per_shift=1, hourly_wage=18.00)
costs_18 = model_18.annual_labor_cost(overtime_hours_pct=0.15)

# Overhead estimate (food, utilities, vehicles, insurance, admin)
annual_overhead = 45_000  # conservative estimate for a 4-bed home

print("=== 4-Bed HCS House P&L at LON 5 ($158/day) ===")
print(f"Revenue: ${annual_revenue:>12,.0f}/yr ({beds} beds × ${lon5_daily}/day × {occupancy:.0%} occupancy)")
print(f"")

for label, costs in [('$13/hr (current)', costs_13), ('$15/hr (living wage)', costs_15), ('$18/hr (Buc-ee\'s)', costs_18)]:
    margin = annual_revenue - costs['total_labor_cost'] - annual_overhead
    margin_pct = margin / annual_revenue * 100
    print(f"  At {label}:")
    print(f"    Labor:    ${costs['total_labor_cost']:>10,.0f} ({costs['ftes']:.1f} FTEs, {costs['ot_hours']:,.0f} OT hrs)")
    print(f"    Overhead: ${annual_overhead:>10,.0f}")
    print(f"    Margin:   ${margin:>10,.0f} ({margin_pct:.1f}%)")
    print()

In [ ]:
# Visualization: House margin at different wage levels
wage_range = np.arange(10.00, 22.01, 0.50)
margins = []

for wage in wage_range:
    model = StaffingModel(residents=4, staff_per_shift=1, hourly_wage=wage)
    costs = model.annual_labor_cost(overtime_hours_pct=0.15)
    margin = annual_revenue - costs['total_labor_cost'] - annual_overhead
    margins.append({
        'wage': wage,
        'margin': margin,
        'margin_pct': margin / annual_revenue * 100,
        'labor_cost': costs['total_labor_cost'],
    })

df_margins = pd.DataFrame(margins)

fig, ax = plt.subplots(figsize=(14, 7))

# Color bars by positive/negative margin
bar_colors = ['#388e3c' if m > 0 else '#d32f2f' for m in df_margins['margin']]
ax.bar(df_margins['wage'], df_margins['margin'], width=0.4, color=bar_colors, edgecolor='white')

ax.axhline(y=0, color='black', linewidth=1.5)

# Mark key wage points
for wage, label, color in [
    (10.60, 'HHSC old\n$10.60', '#d32f2f'),
    (13.00, 'HHSC new\n$13.00', '#f57c00'),
    (15.00, 'Living wage\n$15.00', '#1976d2'),
    (18.00, "Buc-ee's\n$18.00", '#388e3c'),
]:
    ax.axvline(x=wage, color=color, linestyle='--', alpha=0.7, linewidth=1.5)
    ax.text(wage, ax.get_ylim()[1] * 0.85, label, ha='center', fontsize=9, 
            color=color, fontweight='bold')

ax.set_xlabel('Direct Care Worker Hourly Wage ($)', fontsize=13)
ax.set_ylabel('Annual Operating Margin ($)', fontsize=13)
ax.set_title('4-Bed HCS House Operating Margin vs. Worker Wage\n'
             f'LON 5 rate: ${lon5_daily}/day | {occupancy:.0%} occupancy | 15% overtime',
             fontweight='bold', fontsize=14)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Find breakeven
breakeven_row = df_margins.iloc[(df_margins['margin'].abs()).argmin()]
ax.annotate(f'Breakeven ≈ ${breakeven_row["wage"]:.2f}/hr',
            xy=(breakeven_row['wage'], 0), xytext=(breakeven_row['wage'] + 1.5, -15000),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/house_margin_vs_wage.png', bbox_inches='tight')
plt.show()

print(f"\nBreakeven wage: ~${breakeven_row['wage']:.2f}/hr")
print(f"At $13/hr: ${df_margins[df_margins['wage']==13.0]['margin'].iloc[0]:,.0f} margin")
print(f"At $18/hr: ${df_margins[df_margins['wage']==18.0]['margin'].iloc[0]:,.0f} margin")

---
## 3b. ICF/IID: The Institutional Side

ICF/IID is a per-diem (daily) model — Medicaid pays a flat daily rate per resident
for 24-hour supervised care. Rates vary by **Level of Need (LON)** and **facility size**
(Small, Medium, Large).

This is where Council Oaks likely operates.

In [ ]:
# ICF/IID rates from PFD calculator
icf = df_all[df_all['program'] == 'ICF'].copy()
icf['size'] = icf['bill_description'].str.extract(r'(Small|Medium|Large)')
icf['lon'] = icf['bill_description'].str.extract(r'LON (\d+)').astype(int)
icf = icf.sort_values(['size', 'lon'])

print("=== ICF/IID Non-State Community Residential Daily Rates ===\n")
for size in ['Small', 'Medium', 'Large']:
    subset = icf[icf['size'] == size]
    print(f"  {size} Facility:")
    print(f"  {'LON':<6} {'Daily Rate':>12} {'Att Cost':>12} {'Att %':>8} {'Wage/Day':>12}")
    print(f"  {'-'*56}")
    for _, r in subset.iterrows():
        print(f"  LON {r['lon']:<3} ${r['current_rate']:>10.2f} ${r['attendant_cost']:>10.2f} "
              f"{r['attendant_cost_pct']:>7.1%} ${r['wage_required']:>10.2f}")
    print()

---
## 4. Wage Sensitivity: What Would It Cost to Pay a Living Wage?

Using the PFD wage calculator structure: if we change the target wage from $10.60 to $X,
what's the total fiscal impact across all Medicaid attendant services?

In [ ]:
# ICF/IID Small facility — margin vs wage (same chart style as HCS)
# This is the likely model for Council Oaks community homes

icf_rates_small = {
    'LON 1': 155.04,
    'LON 5': 172.68,
    'LON 6': 241.76,
    'LON 8': 196.72,
    'LON 9': 410.92,
}

beds = 4
occupancy = 0.95
overhead = 45_000

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# --- LEFT: ICF vs HCS rate comparison by LON ---
hcs_rates = {'LON 1': 149.08, 'LON 5': 158.00, 'LON 6': 192.80, 'LON 8': 171.04, 'LON 9': 280.24}
lons = ['LON 1', 'LON 5', 'LON 6', 'LON 8', 'LON 9']
x_pos = np.arange(len(lons))

bars_icf = axes[0].bar(x_pos - 0.2, [icf_rates_small[l] for l in lons], 0.35, 
                        label='ICF/IID Small', color='#1976d2', edgecolor='white')
bars_hcs = axes[0].bar(x_pos + 0.2, [hcs_rates[l] for l in lons], 0.35,
                        label='HCS Residential', color='#f57c00', edgecolor='white')

axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(lons)
axes[0].set_xlabel('Level of Need', fontsize=12)
axes[0].set_ylabel('Daily Rate per Resident ($)', fontsize=12)
axes[0].set_title('ICF/IID vs HCS Daily Rates\nby Level of Need', fontweight='bold', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Add value labels
for bar in bars_icf:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3, 
                 f'${bar.get_height():.0f}', ha='center', fontsize=9, fontweight='bold')
for bar in bars_hcs:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                 f'${bar.get_height():.0f}', ha='center', fontsize=9, fontweight='bold')

# --- RIGHT: ICF Small margin vs wage (LON 1 and LON 5) ---
wage_range = np.arange(9.00, 22.01, 0.50)

for lon_label, daily_rate, color, ls in [
    ('LON 1 ($155/day)', 155.04, '#d32f2f', '-'),
    ('LON 5 ($173/day)', 172.68, '#f57c00', '-'),
    ('LON 8 ($197/day)', 196.72, '#1976d2', '--'),
]:
    revenue = daily_rate * beds * occupancy * 365
    margins_icf = []
    for wage in wage_range:
        model = StaffingModel(residents=beds, staff_per_shift=1, hourly_wage=wage)
        costs = model.annual_labor_cost(overtime_hours_pct=0.15)
        margin = revenue - costs['total_labor_cost'] - overhead
        margins_icf.append(margin)
    axes[1].plot(wage_range, margins_icf, label=lon_label, color=color, linewidth=2.5, linestyle=ls)

axes[1].axhline(y=0, color='black', linewidth=1.5)
axes[1].axvline(x=10.50, color='#9c27b0', linestyle=':', linewidth=2, alpha=0.8)
axes[1].text(10.50, axes[1].get_ylim()[0] * 0.15, ' $10.50\n YOUR\n WAGE', 
             fontsize=10, color='#9c27b0', fontweight='bold')
axes[1].axvline(x=10.60, color='#d32f2f', linestyle='--', alpha=0.5)
axes[1].axvline(x=13.00, color='#f57c00', linestyle='--', alpha=0.5)
axes[1].axvline(x=18.00, color='#388e3c', linestyle='--', alpha=0.5)

axes[1].text(10.60, axes[1].get_ylim()[1] * 0.92, 'HHSC\n$10.60', ha='center', fontsize=8, color='#d32f2f')
axes[1].text(13.00, axes[1].get_ylim()[1] * 0.92, 'New\n$13', ha='center', fontsize=8, color='#f57c00')
axes[1].text(18.00, axes[1].get_ylim()[1] * 0.92, "Buc-ee's\n$18", ha='center', fontsize=8, color='#388e3c')

axes[1].set_xlabel('Direct Care Worker Hourly Wage ($)', fontsize=12)
axes[1].set_ylabel('Annual Operating Margin ($)', fontsize=12)
axes[1].set_title('ICF/IID Small Home (4-bed) Operating Margin\nvs. Worker Wage', fontweight='bold', fontsize=14)
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].legend(fontsize=10, loc='upper right')

plt.tight_layout()
plt.savefig('../reports/icf_rates_and_margins.png', bbox_inches='tight')
plt.show()

In [ ]:
# For each service line, calculate what the rate WOULD be at different wage targets
# Key relationship from PFD calculator:
#   attendant_cost = wage_required * (1 + PTB%)  where PTB = 10.75%
#   attendant_cost_pct = attendant_cost / current_rate

PTB = 0.1075  # payroll taxes & benefits percentage from PFD calculator
CURRENT_TARGET = 10.60

# Filter to services with valid hourly wage data
df_fiscal = df_all[
    (df_all['wage_required'] > 0) & 
    (df_all['attendant_cost_pct'] > 0) &
    (df_all['sfy26_est_hours'].notna()) &
    (df_all['sfy26_est_hours'] > 0)
].copy()

print(f"Service lines with fiscal data: {len(df_fiscal)}")
print(f"Total estimated attendant hours SFY 2026: {df_fiscal['sfy26_est_hours'].sum():,.0f}")
print(f"Total estimated attendant hours SFY 2027: {df_fiscal['sfy27_est_hours'].sum():,.0f}")

In [ ]:
# THE UNCOMFORTABLE MATH: Where does the margin go at $10.50/hr?
# At your wage, the house has significant margin — you're subsidizing the operation.

print("=" * 80)
print("WHERE DOES THE MONEY GO? — ICF Small, 4-bed, $10.50/hr worker wage")
print("=" * 80)

model_you = StaffingModel(residents=4, staff_per_shift=1, hourly_wage=10.50)
costs_you = model_you.annual_labor_cost(overtime_hours_pct=0.15)
your_annual = 10.50 * 2080  # 40hr/wk, 52 weeks, no OT

for lon, daily_rate in icf_rates_small.items():
    revenue = daily_rate * beds * occupancy * 365
    margin = revenue - costs_you['total_labor_cost'] - overhead
    
    print(f"\n  {lon} (${daily_rate}/day per resident):")
    print(f"    Medicaid pays the provider:    ${revenue:>10,.0f}/yr")
    print(f"    Total labor cost (all staff):  ${costs_you['total_labor_cost']:>10,.0f}/yr")
    print(f"    Overhead:                      ${overhead:>10,.0f}/yr")
    print(f"    Remaining margin:              ${margin:>10,.0f}/yr  ({margin/revenue:.1%})")
    print(f"    ─────────────────────────────────────")
    print(f"    YOUR take-home (pre-tax):      ${your_annual:>10,.0f}/yr")
    print(f"    YOUR share of revenue:                     {your_annual/revenue:.1%}")
    if margin > your_annual:
        print(f"    ⚠ The margin ALONE (${margin:,.0f}) exceeds your annual pay (${your_annual:,.0f})")

print(f"\n{'=' * 80}")
print(f"You earn ${10.50:.2f}/hr = ${your_annual:,.0f}/yr before taxes.")
print(f"That's ${your_annual/12:,.0f}/month. Federal poverty line for a single person: $15,060/yr.")
print(f"You are earning {your_annual/15060:.1f}x the poverty line for work that requires")
print(f"24/7 vigilance, medication administration, crisis response, and documentation.")

In [ ]:
# Model fiscal impact at different target wages
# For each service: additional cost = (new_wage - current_wage_in_rate) * (1+PTB) * estimated_hours

target_wages = [10.60, 12.00, 13.00, 14.00, 15.00, 16.00, 17.00, 18.00]
fiscal_impacts = []

for target in target_wages:
    total_af = 0  # all funds
    total_gr = 0  # general revenue (state share)
    
    for _, svc in df_fiscal.iterrows():
        current_wage = svc['wage_required']
        hours_26 = svc['sfy26_est_hours']
        hours_27 = svc['sfy27_est_hours']
        
        if target > current_wage:
            wage_increase = target - current_wage
            cost_increase_per_hour = wage_increase * (1 + PTB)
            annual_impact = cost_increase_per_hour * (hours_26 + hours_27)  # biennium
            total_af += annual_impact
            total_gr += annual_impact * 0.406  # state share ~40.6%
    
    fiscal_impacts.append({
        'target_wage': target,
        'biennium_impact_af': total_af,
        'biennium_impact_gr': total_gr,
        'biennium_impact_ff': total_af - total_gr,  # federal share
    })

df_fiscal_impact = pd.DataFrame(fiscal_impacts)

print("=== Estimated Biennium Fiscal Impact by Target Wage ===")
print(f"{'Target Wage':>12} {'All Funds':>16} {'State (GR)':>16} {'Federal':>16}")
print('-' * 65)
for _, row in df_fiscal_impact.iterrows():
    print(f"  ${row['target_wage']:>8.2f}/hr  ${row['biennium_impact_af']:>14,.0f}  "
          f"${row['biennium_impact_gr']:>14,.0f}  ${row['biennium_impact_ff']:>14,.0f}")

In [ ]:
# Visualize fiscal impact
fig, ax = plt.subplots(figsize=(14, 7))

# Stacked bar: state vs federal share
x = df_fiscal_impact['target_wage']
gr = df_fiscal_impact['biennium_impact_gr'] / 1e6  # millions
ff = df_fiscal_impact['biennium_impact_ff'] / 1e6

ax.bar(x, gr, width=0.6, label='State Share (GR, ~40.6%)', color='#d32f2f', edgecolor='white')
ax.bar(x, ff, width=0.6, bottom=gr, label='Federal Share (FF, ~59.4%)', color='#1976d2', edgecolor='white')

# Labels on top
for i, (wage, total) in enumerate(zip(x, df_fiscal_impact['biennium_impact_af'] / 1e6)):
    if total > 0:
        ax.text(wage, total + 5, f'${total:.0f}M', ha='center', fontweight='bold', fontsize=10)

ax.set_xlabel('Target Attendant Hourly Wage ($)', fontsize=13)
ax.set_ylabel('Biennium Fiscal Impact ($ Millions)', fontsize=13)
ax.set_title('Cost to Raise Direct Care Wages Across All TX Medicaid Attendant Services\n'
             'Estimated biennium impact (SFY 2026-2027) by target wage',
             fontweight='bold', fontsize=14)
ax.legend(fontsize=11)
ax.set_xticks(target_wages)
ax.set_xticklabels([f'${w:.2f}' for w in target_wages])

plt.tight_layout()
plt.savefig('../reports/fiscal_impact_by_target_wage.png', bbox_inches='tight')
plt.show()

---
## 5. The Rate Anatomy: Where Does Each Dollar Go?

For HCS PAS/HAB (the most common personal care service), break down the rate.

In [ ]:
# HCS PAS/HAB rate breakdown
pas_hab = df_target[
    (df_target['program'] == 'HCS') & 
    (df_target['service_description'] == 'PAS/HAB (CFC) and Transportation') &
    (~df_target['is_cds'])
].iloc[0]

rate = pas_hab['current_rate']  # $18.36/hr
att_cost = pas_hab['attendant_cost']  # $15.15
non_att = rate - att_cost  # admin/overhead built into rate
wage_in_rate = pas_hab['wage_required']  # $13.69
ptb_in_rate = att_cost - wage_in_rate  # payroll taxes & benefits

print(f"=== HCS PAS/HAB (CFC) Rate Anatomy ===")
print(f"Total Medicaid Rate:              ${rate:.2f}/hr")
print(f"  ├── Attendant wage component:   ${wage_in_rate:.2f} ({wage_in_rate/rate:.1%})")
print(f"  ├── Payroll taxes & benefits:   ${ptb_in_rate:.2f} ({ptb_in_rate/rate:.1%})")
print(f"  └── Admin/overhead/margin:      ${non_att:.2f} ({non_att/rate:.1%})")
print(f"")
print(f"But the ACTUAL wage paid may be lower than ${wage_in_rate:.2f}")
print(f"because providers absorb costs the rate doesn't fully cover.")

# Pie chart
fig, ax = plt.subplots(figsize=(8, 8))
sizes = [wage_in_rate, ptb_in_rate, non_att]
labels = [
    f'Worker Wage\n${wage_in_rate:.2f} ({wage_in_rate/rate:.1%})',
    f'Payroll Tax/Benefits\n${ptb_in_rate:.2f} ({ptb_in_rate/rate:.1%})',
    f'Admin/Overhead/Margin\n${non_att:.2f} ({non_att/rate:.1%})',
]
colors_pie = ['#1976d2', '#f57c00', '#757575']
explode = (0.05, 0, 0)

ax.pie(sizes, labels=labels, colors=colors_pie, explode=explode,
       autopct='', startangle=90, textprops={'fontsize': 12})
ax.set_title(f'Where Does Each Dollar of HCS PAS/HAB Rate Go?\n'
             f'Total rate: ${rate:.2f}/hr', fontweight='bold', fontsize=14)

plt.tight_layout()
plt.savefig('../reports/rate_anatomy_hcs_pashab.png', bbox_inches='tight')
plt.show()

---
## 7. Key Takeaways

1. **The $10.60 is real** — it's literally cell B7 in HHSC's own published spreadsheet
2. **The BLS market mean for TX Home Health Aides is $12.19/hr** — below even the "new" $13 target
3. **Buc-ee's pays $5.81/hr more** than the HHSC market mean for life-or-death care work
4. **The rate structure caps what providers can pay** — 82.5% of HCS PAS/HAB goes to attendant costs, leaving 17.5% for everything else
5. **At $18/hr (competitive), both HCS and ICF LON 1-5 houses go negative** — the funding model structurally cannot support market wages
6. **ICF rates are slightly higher than HCS** ($155 vs $149/day at LON 1) but both trap workers below $15/hr
7. **At $10.50/hr, the house margin exceeds a single worker's annual pay** at LON 5+ — the money exists in the rate, it just doesn't reach the worker
8. **~315,000 workers** in Texas are in this wage band (BLS SOC 31-1120)

In [ ]:
# Summary stats for citation
print("=== Citation-Ready Stats ===")
print(f"HHSC default target wage: $10.60/hr (PFD Wage Calculator, updated 2/7/2025)")
print(f"HHSC new target wage: $13.00/hr (KERA, 2025-03-04)")
print(f"BLS TX Home Health Aide mean: $12.19/hr (OEWS May 2024)")
print(f"BLS TX Home Health Aide median: $11.29/hr (OEWS May 2024)")
print(f"BLS TX Home Health Aide 25th pctile: $10.62/hr — matches HHSC $10.60")
print(f"BLS TX Home Health Aide employment: 314,610 (OEWS May 2024)")
print(f"BLS TX Nursing Assistant mean: $17.79/hr (OEWS May 2024)")
print(f"Buc-ee's entry wage: $18.00/hr (public job postings)")
print(f"")
print(f"=== HCS ===")
print(f"HCS PAS/HAB rate: $18.36/hr, of which $13.69 (74.6%) is wage component")
print(f"HCS Residential LON 5 rate: $158.00/day")
print(f"HCS 4-bed house breakeven: ~$15.50/hr")
print(f"")
print(f"=== ICF/IID ===")
print(f"ICF/IID Small LON 1 rate: $155.04/day")
print(f"ICF/IID Small LON 5 rate: $172.68/day")
print(f"ICF/IID Small LON 9 rate: $410.92/day")
print(f"ICF/IID Small 4-bed LON 1 breakeven: ~$15.00/hr")
print(f"ICF/IID Small 4-bed LON 5 breakeven: ~$16.50/hr")
print(f"At $10.50/hr, LON 5 margin ($75K) > worker annual pay ($21.8K)")
print(f"")
print(f"FMAP: ~40.6% state / ~59.4% federal (SFY 2026)")